# 1 - Data Pipeline
Pulls and caches raw data from all sources, and does the schema exploratrion needed before building out the later notebooks.
### Sources:
* StatsBomb(event-level, live in-match features)
* Understat (season teamxG/xGA, pre-match prior)
* FBref (player-season stats, clustering)
* football-data.co.uk (betting odds, market comparison)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import(list_available_competitions,
    load_statsbomb_matches,
    load_statsbomb_events_for_competition,
    load_understat_team_data,
    load_understat_team_data_multi,
    load_fbref_player_stats,
    _read_stat_type_with_retry,
    _flatten_and_prefix_columns,
    load_fbref_player_stats_all,
    load_football_data_odds,
    load_football_data_odds_multi)

## StatsBomb 1. - check available competitions
Check match counts before comitting to a competition/season

In [2]:
comps = list_available_competitions()
comps.head(10)

C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,None,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,None,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,None,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,None,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,None,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,None,2024-02-12T12:47:34.340413


In [3]:
match_counts = []

for _, row in comps.iterrows():
    comp_id = row['competition_id']
    season_id = row['season_id']
    try:
        matches = load_statsbomb_matches(competition_id=comp_id, season_id=season_id)
        count = len(matches)
    except Exception as e:
        count = None  # some competition/season combos error out or have no data

    match_counts.append({
        'competition_name': row['competition_name'],
        'season_name': row['season_name'],
        'competition_id': comp_id,
        'season_id': season_id,
        'match_count': count
    })

match_counts_df = pd.DataFrame(match_counts)
match_counts_df.sort_values('match_count', ascending=False).head(20)

C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbomb

,competition_name,season_name,competition_id,season_id,match_count
70,Serie A,2015/2016,12,27,380
45,La Liga,2015/2016,11,27,380
68,Premier League,2015/2016,2,27,380
63,Ligue 1,2015/2016,7,27,377
58,Liga F,2023/2024,182,281,240
66,NWSL,2023,49,107,137
38,Frauen Bundesliga,2023/2024,135,281,132
25,FA Women's Super League,2023/2024,37,281,132
26,FA Women's Super League,2020/2021,37,90,131
72,Serie A Women,2023/2024,131,281,130


In [4]:
# Confirmed full-coverage options:
#   La Liga 2015/2016:        competition_id=11, season_id=27  -> 380 matches
#   Premier League 2015/2016: competition_id=2,  season_id=27  -> 380 matches
#   Serie A 2015/2016:        competition_id=12, season_id=27  -> 380 matches
# Same season across three leagues = 1,140 matches total, consistent tagging/xG methodology.
# (World Cup and Europa League were considered and dropped -- see data_loader.py docstring for why.)

laliga_matches = load_statsbomb_matches(competition_id=11, season_id=27)
prem_matches = load_statsbomb_matches(competition_id=2, season_id=27)
seriea_matches = load_statsbomb_matches(competition_id=12, season_id=27)

laliga_matches['league'] = 'La Liga'
prem_matches['league'] = 'Premier League'
seriea_matches['league'] = 'Serie A'

match_list = pd.concat([laliga_matches, prem_matches, seriea_matches], ignore_index=True)
match_list_to_save = match_list[['match_id', 'home_team', 'away_team', 'home_score', 'away_score', 'match_date', 'league']]
match_list_to_save.to_parquet('../data/raw/statsbomb_match_list.parquet')
print(len(match_list))
print('La Liga 2015/16 matches:', len(laliga_matches))
print('Premier League 2015/16 matches:', len(prem_matches))
print('Serie A 2015/16 matches:', len(seriea_matches))

laliga_matches[['match_id','home_team','away_team','home_score','away_score','match_date']].head()

C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


1140
La Liga 2015/16 matches: 380
Premier League 2015/16 matches: 380
Serie A 2015/16 matches: 380


C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,home_team,away_team,home_score,away_score,match_date
0,3825739,Real Madrid,Sporting Gijón,5,1,2016-01-17
1,3825848,Levante UD,Eibar,2,2,2015-09-23
2,3825895,Las Palmas,Sevilla,2,0,2015-09-23
3,3825894,RC Deportivo La Coruña,Getafe,0,2,2016-05-01
4,3825855,Málaga,Levante UD,3,1,2016-05-02


## Statsbomb 2. - pull and cache full event data
Slow step (one API call per match), load_statsbomb_events_for_competition caches the result to data/raw/ as parquet, so re-running this cell after the first time just loads from the disk.

In [5]:
# Pull all 3 leagues
laliga_events = load_statsbomb_events_for_competition(competition_id=11, season_id=27)
prem_events = load_statsbomb_events_for_competition(competition_id=2, season_id=27)
seriea_events = load_statsbomb_events_for_competition(competition_id=12, season_id=27)

print('La Liga events:', len(laliga_events))
print('Premier League events:', len(prem_events))
print('Serie A events:', len(seriea_events))

La Liga events: 1295354
Premier League events: 1313773
Serie A events: 1353739


In [6]:
# Combine into one dataframe for downstream feature engineering, tag the leagues for filtering later on
laliga_events['league'] = 'La Liga'
prem_events['league'] = 'Premier League'
seriea_events['league'] = 'Serie A'

all_events = pd.concat([laliga_events, prem_events, seriea_events], ignore_index = True)
all_events.to_parquet('../data/raw/statsbomb_events_all.parquet')
print(len(all_events))
print(all_events.duplicated(subset=['id']).sum())
print(all_events['league'].value_counts())

3962866
0
league
Serie A           1353739
Premier League    1313773
La Liga           1295354
Name: count, dtype: int64


Ligue 1 was added specifically for the player-clustering dataset (Part B), not the win-probability model (Part A), to avoid re-validating the already-complete Elo/live-model pipeline — a deliberate scope decision, not an oversight.

In [7]:
ligue1_matches = load_statsbomb_matches(competition_id=7, season_id=27)
ligue1_matches['league'] = 'Ligue 1'
ligue1_events = load_statsbomb_events_for_competition(competition_id=7, season_id=27)
ligue1_events['league'] = 'Ligue 1'

ligue1_matches.to_parquet('../data/raw/statsbomb_ligue1_matches.parquet')
ligue1_events.to_parquet('../data/raw/statsbomb_ligue1_events.parquet')
print(len(ligue1_matches), len(ligue1_events))

C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\ausuc\anaconda3\Lib\site-packages\statsbomb

377 1358593


## Understat 3. - Pre-match team strength data

Needs the soccerdata package. Pull a few seasons across leagues to get enough matches for a rolling xG-differential

In [8]:
understat_data = load_understat_team_data_multi(
    leagues=['ENG-Premier League', 'ESP-La Liga', 'ITA-Serie A'],
    seasons=['2014-2015', '2015-2016']
)
understat_data.to_parquet('../data/raw/understat_team_data.parquet')
print('Total Understat team-match rows:', len(understat_data))
understat_data.head()

[09/29/26 14:48:26] INFO     No custom team name replacements found. You can configure these in       ]8;id=4722806;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=4722807;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\ausuc\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=4722813;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=4722814;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\ausuc\soccerdata\config\league_dict.json.                                    

                    INFO     Saving cached data to C:\Users\ausuc\soccerdata\data\Understat          ]8;id=4722821;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=4722822;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-29 14:48:26] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\ausuc\anaconda3\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=4722829;file://C:\Users\ausuc\anaconda3\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=4722830;file://C:\Users\ausuc\anaconda3\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\ausuc\anaconda3\Lib\site-packages\tls_requests\bin\tls-clien                 
                             t-xgo-1.13.1-windows-amd64.dll                                                        

[09/29/26 14:48:27] INFO     Saving cached data to C:\Users\ausuc\soccerdata\data\Understat          ]8;id=4722835;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=4722836;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[09/29/26 14:48:28] INFO     Saving cached data to C:\Users\ausuc\soccerdata\data\Understat          ]8;id=4722841;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=4722842;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Total Understat team-match rows: 2280


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,away_team_code,home_team_code,...,away_deep_completions,home_points,home_expected_points,home_goals,home_xg,home_np_xg,home_np_xg_difference,home_ppda,home_deep_completions,league
0,1,2014,4755,2014-08-16 17:30:00,83,78,Arsenal,Crystal Palace,CRY,ARS,...,0,3,2.7946,2,1.55411,1.55411,1.395959,2.933333,9,ENG-Premier League
1,1,2014,4750,2014-08-16 15:00:00,75,72,Leicester,Everton,EVE,LEI,...,15,1,1.9461,2,1.2783,1.2783,0.665027,14.48,1,ENG-Premier League
2,1,2014,4749,2014-08-16 12:45:00,89,84,Manchester United,Swansea,SWA,MUN,...,2,0,2.2359,1,1.16635,1.16635,0.888274,10.12,8,ENG-Premier League
3,1,2014,4751,2014-08-16 15:00:00,202,91,Queens Park Rangers,Hull,HUL,QPR,...,5,0,2.0149,0,1.90067,1.12689,0.00932,12.823529,1,ENG-Premier League
4,1,2014,4752,2014-08-16 15:00:00,85,71,Stoke,Aston Villa,AVL,STO,...,4,0,0.8041,0,0.423368,0.423368,-0.486406,4.125,3,ENG-Premier League


## FBref 4. - player-season stats for clustering

Pull each stat_type separately (standard, shooting, passing, defense, possesion) and join on player+team+season.

In [9]:
# Single 'Big 5' combined pull (Premier League, La Liga, Serie A, Bundesliga, Ligue 1)
# instead of looping per-league -- ITA-Serie A consistently failed its own season
# lookup when pulled individually, and this also means one browser session instead
# of several, which sidesteps the earlier Selenium connection flakiness.
# Keeping all 5 leagues deliberately: more players for clustering, and Part B doesn't
# need to match Part A's exact 3-league set.

fbref_players = load_fbref_player_stats_all(season='2023-24')
fbref_players.to_parquet('../data/raw/fbref_players_data.parquet')

print('Total FBref player-season rows:', len(fbref_players))
print('Leagues included:', fbref_players['league'].unique())
fbref_players.head()

[09/29/26 14:48:29] INFO     Saving cached data to C:\Users\ausuc\soccerdata\data\FBref              ]8;id=4722847;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=4722848;file://C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[09/29/26 14:48:33] WARNING  C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\fbref.py:171:    ]8;id=4722855;file://C:\Users\ausuc\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=4722856;file://C:\Users\ausuc\anaconda3\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of DataFrame concatenation with empty or                  
                             all-NA entries is deprecated. In a future version, this will no longer                
                             exclude empty or all-NA columns when determining the result dtypes. To                
                             retain the old behavior, exclude the relevant entries before the                      
                             concat operation.                                                                     
                               pd.concat(dfs)                                                                      
                                                                                                                   

[09/29/26 14:48:35] WARNING  C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\fbref.py:171:    ]8;id=4722861;file://C:\Users\ausuc\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=4722862;file://C:\Users\ausuc\anaconda3\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of DataFrame concatenation with empty or                  
                             all-NA entries is deprecated. In a future version, this will no longer                
                             exclude empty or all-NA columns when determining the result dtypes. To                
                             retain the old behavior, exclude the relevant entries before the                      
                             concat operation.                                                                     
                               pd.concat(dfs)                                                                      
                                                                                                                   

[09/29/26 14:48:36] WARNING  C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\fbref.py:171:    ]8;id=4722867;file://C:\Users\ausuc\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=4722868;file://C:\Users\ausuc\anaconda3\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of DataFrame concatenation with empty or                  
                             all-NA entries is deprecated. In a future version, this will no longer                
                             exclude empty or all-NA columns when determining the result dtypes. To                
                             retain the old behavior, exclude the relevant entries before the                      
                             concat operation.                                                                     
                               pd.concat(dfs)                                                                      
                                                                                                                   

[09/29/26 14:48:38] WARNING  C:\Users\ausuc\anaconda3\Lib\site-packages\soccerdata\fbref.py:171:    ]8;id=4722873;file://C:\Users\ausuc\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=4722874;file://C:\Users\ausuc\anaconda3\Lib\warnings.py#110\110]8;;\
                             FutureWarning: The behavior of DataFrame concatenation with empty or                  
                             all-NA entries is deprecated. In a future version, this will no longer                
                             exclude empty or all-NA columns when determining the result dtypes. To                
                             retain the old behavior, exclude the relevant entries before the                      
                             concat operation.                                                                     
                               pd.concat(dfs)                                                                      
                                                                                                                   

Total FBref player-season rows: 3513
Leagues included: ['ENG-Premier League' 'ESP-La Liga' 'FRA-Ligue 1' 'ITA-Serie A' nan]


,league,season,team,player,standard_nation,standard_pos,standard_age,standard_born,standard_Playing_Time_MP,standard_Playing_Time_Starts,...,misc_Performance_2CrdY,misc_Performance_Fls,misc_Performance_Fld,misc_Performance_Off,misc_Performance_Crs,misc_Performance_Int,misc_Performance_TklW,misc_Performance_PKwon,misc_Performance_PKcon,misc_Performance_OG
0,ENG-Premier League,2324,Arsenal,Aaron Ramsdale,ENG,GK,25,1998,6,6,...,0,0,1,0,0,0,0,<NA>,<NA>,0
1,ENG-Premier League,2324,Arsenal,Ben White,ENG,DF,25,1997,37,35,...,0,19,12,11,57,30,28,<NA>,<NA>,0
2,ENG-Premier League,2324,Arsenal,Bukayo Saka,ENG,FW,21,2001,35,35,...,0,41,69,8,181,16,37,<NA>,<NA>,0
3,ENG-Premier League,2324,Arsenal,Cédric Soares,POR,"DF,MF",31,1991,3,0,...,0,0,1,0,4,0,0,<NA>,<NA>,0
4,ENG-Premier League,2324,Arsenal,David Raya,ESP,GK,27,1995,32,32,...,0,0,4,0,0,0,1,<NA>,<NA>,0


## football-data.co.uk 5.- betting odds
direct CSV pull, no package needed. E0 = Premier League; check football-data.uk for other leagues

In [10]:
odds_data = load_football_data_odds_multi(
    league_codes=['E0', 'SP1', 'I1'],   # Premier League, La Liga, Serie A
    season_codes=['1415', '1516']       # 2024/2025 and 2025/2026, both complete seasons
)
odds_data.to_parquet('../data/raw/odds_data.parquet')
print('Total odds rows:', len(odds_data))
print(odds_data[['league_code','season_code']].value_counts())
odds_data[['Date','HomeTeam','AwayTeam','FTHG','FTAG','B365H','B365D','B365A','league_code','season_code']].head()

Total odds rows: 2283
league_code  season_code
E0           1415           381
I1           1415           381
             1516           381
E0           1516           380
SP1          1415           380
             1516           380
Name: count, dtype: int64


,Date,HomeTeam,AwayTeam,FTHG,FTAG,B365H,B365D,B365A,league_code,season_code
0,16/08/14,Arsenal,Crystal Palace,2.0,1.0,1.25,6.5,15.0,E0,1415
1,16/08/14,Leicester,Everton,2.0,2.0,3.20,3.4,2.4,E0,1415
2,16/08/14,Man United,Swansea,1.0,2.0,1.36,5.0,11.0,E0,1415
3,16/08/14,QPR,Hull,0.0,1.0,2.50,3.3,3.1,E0,1415
4,16/08/14,Stoke,Aston Villa,0.0,1.0,1.95,3.5,4.5,E0,1415


## Next Steps:
- `02_prematch_prior_tool.ipynb` — build `get_prematch_prior()` from the Understat data
- `03_live_update_model.ipynb` — build `get_live_win_prob()` / `get_live_fair_odds()` from the StatsBomb events
- `04_market_comparison.ipynb` — join odds onto the pre-match tool's output
- `05_player_clustering.ipynb` — PCA + clustering on the FBref stats